In [ ]:
# Import the libraries we need for loading images, building the model, and plotting results.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from IPython.display import display
from tensorflow.keras.preprocessing.image import ImageDataGenerator

tf.__version__


In [ ]:
# Point this to the folder that contains the train, valid, and test subfolders.
dataset_root = Path('waste_dataset')

train_dir = dataset_root / 'train'
valid_dir = dataset_root / 'valid'
test_dir = dataset_root / 'test'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 5
LEARNING_RATE = 1e-3
FINE_TUNE_LR = 1e-5
SEED = 42

for folder in [train_dir, valid_dir, test_dir]:
    if not folder.exists():
        raise FileNotFoundError(f'Missing expected folder: {folder.resolve()}')

dataset_root.resolve()


In [ ]:
# Add augmentation to the training set so the model sees more variety while learning.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation and test images only need to be rescaled.
valid_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode='categorical',
    shuffle=False
)

# Keep the class names so we can interpret predictions later.
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

len(train_generator), class_names, num_classes


In [ ]:
# Build the transfer learning model using MobileNetV2 as the feature extractor.
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

extract_feat_model = models.Model(inputs, outputs, name='extract_features_model')

extract_feat_model.summary()


In [ ]:
extract_feat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

extract_feat_model.metrics_names


In [ ]:
# First train only the new classification head while the base model stays frozen.
history_extract = extract_feat_model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=INITIAL_EPOCHS
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_extract.history['accuracy'], label='Train Accuracy')
plt.plot(history_extract.history['val_accuracy'], label='Validation Accuracy')
plt.title('Extract Features Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Unfreeze part of the base model so we can fine-tune higher-level features.
base_model.trainable = True
fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

fine_tune_model = extract_feat_model

fine_tune_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_model.summary()


In [ ]:
# Continue training with a smaller learning rate to fine-tune the model.
history_fine_tune = fine_tune_model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
    initial_epoch=history_extract.epoch[-1] + 1
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_fine_tune.history['loss'], label='Train Loss')
plt.plot(history_fine_tune.history['val_loss'], label='Validation Loss')
plt.title('Fine-Tuned Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_fine_tune.history['accuracy'], label='Train Accuracy')
plt.plot(history_fine_tune.history['val_accuracy'], label='Validation Accuracy')
plt.title('Fine-Tuned Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Helper function to show one test image and compare the true label with the prediction.
def plot_test_prediction(model, generator, class_names, index_to_plot=1, title='Prediction'):
    image_batch, label_batch = generator[index_to_plot]

    image = image_batch[0]
    true_label_index = int(np.argmax(label_batch[0]))
    prediction = model.predict(image_batch, verbose=0)[0]
    pred_label_index = int(np.argmax(prediction))

    display_image = (image + 1.0) / 2.0
    display_image = np.clip(display_image, 0, 1)

    plt.figure(figsize=(5, 5))
    plt.imshow(display_image)
    plt.axis('off')
    plt.title(
        f"{title}\nTrue: {class_names[true_label_index]} | Pred: {class_names[pred_label_index]}\nConfidence: {prediction[pred_label_index]:.4f}"
    )
    plt.show()


In [ ]:
plot_test_prediction(
    extract_feat_model,
    test_generator,
    class_names,
    index_to_plot=1,
    title='Extract Features Model Test Prediction'
)

plot_test_prediction(
    fine_tune_model,
    test_generator,
    class_names,
    index_to_plot=1,
    title='Fine-Tuned Model Test Prediction'
)
